# Inverse Student Simulation — Interactive Tutorial

This notebook is a **five-minute, no-GPU, no-API-key** walkthrough of the
core ideas behind the *Inverse Student Simulation* (ISS) framework. It is
meant to help a reader understand **how the paper's pipeline works and
why its results look the way they do**, before diving into the full
`scripts/` reproduction pipeline.

Everything below calls the **real library code** in `src/iss` (not a
toy re-implementation) on three tiny synthetic dialogues, so what you see
here is mechanically the same machinery used at full scale on the
394-dialogue MathDial test set — just at a scale you can read line by line.

**What this notebook shows:**

1. The fixed structured-state schema `Z = (m, C, g)` and how malformed
   LLM output gets deterministically repaired into it.
2. How **Pseudo-Z** (evidence-free heuristic) and **Random-Z** (uniform
   control) are constructed from a dialogue prefix.
3. How KC-mastery recovery is scored (Brier, ECE, within-dialogue
   Spearman, top-3 overlap) — the same metrics behind the paper's E1 /
   KC-structure tables.
4. **The central misconception-facet finding of the paper**: why
   near-perfect F1@5/MRR can be a label-uniformity artifact rather than
   evidence of genuine per-dialogue discrimination.

**What this notebook is *not***: it does not reproduce the paper's
reported numbers — those require the real MathDial corpus, LLM-generated
silver labels, and a fine-tuned `Qwen2.5-3B-Instruct` inverter/forward
simulator. For that, see the top-level `README.md`, section
"Reproducing Main Results", and `results/expected_metrics.json` for the
target numbers to compare against.

> Prefer plain scripts? Each cell below has a 1:1 counterpart in
> `tutorial/0N_*.py`, runnable standalone with `python tutorial/0N_*.py`.

In [ ]:
# Setup: make the real `src/iss` library importable from this notebook.
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "tutorial" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
from scipy.stats import spearmanr

print(f"Repository root: {REPO_ROOT}")

## 1. The structured latent state `Z = (m, C, g)`

Every dialogue is inverted into a fixed-shape JSON object with three parts:

- **`m`** — mastery probabilities over 30 fixed knowledge components (`KC01`–`KC30`)
- **`C`** — activation probabilities over 70 fixed misconception codes (`M001`–`M070`)
- **`g`** — four exploratory metacognitive scalars (monitoring accuracy, help-seeking
  ratio, confidence–correctness gap, hint uptake)

Because `Z` is generated autoregressively by an LLM, raw model output is rarely
perfectly well-formed (short/aliased keys, missing entries, extra fields). The
`iss.schema` module applies **deterministic repair** before validating against
the Pydantic/JSON schema — no LLM call is involved in this step, it is pure
post-processing.

In [ ]:
from iss.schema.grammar import validate_latent_z_json
from iss.schema.kc_ontology import get_kc_ids, load_knowledge_components
from iss.schema.misconception_catalogue import get_misconception_ids
from iss.schema.repair import repair_latent_z_json

kc_ids = get_kc_ids()
misc_ids = get_misconception_ids()
print(f"{len(kc_ids)} knowledge components (expect 30): {kc_ids[:5]} ...")
print(f"{len(misc_ids)} misconception codes (expect 70): {misc_ids[:5]} ...")

kcs = load_knowledge_components()
print(f"\nExample KC: {kcs[0].id} — {kcs[0].name}\n  {kcs[0].description}")

In [ ]:
# A deliberately non-canonical model output: short/aliased keys and missing
# entries -- exactly the kind of drift a fine-tuned LLM produces in practice.
raw_model_output = {
    "mastery": {f"KC{i}": 0.5 for i in range(1, 30)},  # short keys, KC30 missing
    "misconceptions": {"M1": 0.4, "M7": 0.1, "M23": 0.05},  # short keys, most missing
    "metacog": {
        "monitoring": 0.4,      # alias for monitoring_accuracy
        "help-seeking": 0.2,    # alias for help_seeking_ratio
        "conf_gap": 0.1,        # alias for confidence_correctness_gap
        "hint": 0.3,            # alias for hint_uptake
    },
    "unexpected_field": "should be dropped",
}

repaired = repair_latent_z_json(raw_model_output)
z = validate_latent_z_json(repaired)
print(f"Recovered {len(z.mastery.values)} KC entries and "
      f"{len(z.misconceptions.probs)} misconception entries after repair.")
print("\nRepaired metacog scalars:", z.metacog.model_dump())

## 2. Toy dialogues, Pseudo-Z, and Random-Z

`tutorial/toy_dialogues.json` contains three synthetic tutor–student dialogues
(clearly **not** MathDial — self-authored for this tutorial). We build two
kinds of state for each, using the exact functions in `iss.forward.pseudo_z`
that the full pipeline uses:

- **Pseudo-Z** — an evidence-free lexical heuristic (regex cues like
  "fraction", "equation", hedging phrases) applied to the dialogue prefix.
  It is used both as an early-bootstrap weak label *and*, importantly, as the
  **label-prior baseline** in the paper's misconception sparsity audit
  (Section 4 below).
- **Random-Z** — IID `Uniform[0,1]` draws per field. This is the counterfactual
  control arm in the paper's forward-replay experiments (E4): if a forward
  model's next-turn predictions are equally likely under Random-Z as under the
  real (silver) Z, the model is not actually using the structured state.

In [ ]:
from iss.forward.pseudo_z import pseudo_latent_z_from_prefix, uniform_random_latent_z
from iss.schema.latent import DialogueTurn

toy_dialogues = json.loads((Path("toy_dialogues.json") if Path("toy_dialogues.json").exists()
                             else REPO_ROOT / "tutorial" / "toy_dialogues.json").read_text(encoding="utf-8"))


def to_turns(raw_turns):
    return [DialogueTurn(turn_index=i, speaker=t["speaker"], text=t["text"])
            for i, t in enumerate(raw_turns)]


states = []
for dlg in toy_dialogues:
    print(f"\n=== {dlg['dialogue_id']} ===")
    print(f"Q: {dlg['question']}")
    for t in dlg["turns"]:
        print(f"  {t['speaker']:>8}: {t['text']}")

    turns = to_turns(dlg["turns"])
    pseudo_z = pseudo_latent_z_from_prefix(turns)
    random_z = uniform_random_latent_z(seed=1000 + len(states))
    n_shifted = sum(1 for v in pseudo_z.mastery.values.values() if abs(v - 0.5) > 1e-9)
    print(f"  -> Pseudo-Z: {n_shifted}/30 KCs shifted from the 0.5 prior by lexical cues; "
          f"help_seeking_ratio={pseudo_z.metacog.help_seeking_ratio:.2f}")
    states.append({"dialogue_id": dlg["dialogue_id"], "pseudo_z": pseudo_z, "random_z": random_z})

## 3. Scoring KC-mastery recovery

The paper's E1 / KC-structure analysis distinguishes two very different
questions, and this distinction is the key to understanding why its headline
numbers look contradictory at first glance:

- **Pooled, cross-dialogue discrimination (AUC)** — across *all* dialogues,
  does a higher predicted mastery score correspond to a KC the student
  actually mastered? On the real 394-dialogue test set this is **near
  chance (AUC ≈ 0.506)**.
- **Within-dialogue structure (Spearman correlation, top-3 overlap)** — *within
  a single dialogue's own 30-KC profile*, does the model get the *relative
  ordering* of strong vs. weak KCs right? On the real test set this is
  **positive and significant (ρ ≈ 0.274, top-3 overlap ≈ 40.7%** vs. 33%
  chance).

Below we compute the same four metrics (Brier, ECE, within-dialogue Spearman,
top-3 overlap) on the toy dialogues, using a seeded random draw as a
stand-in "reference" state (there are no real annotations at this toy scale —
this cell demonstrates the *metric machinery*, not the paper's numbers).

In [ ]:
from iss.eval.metrics import binary_brier, ece


def toy_reference_z(seed):
    """Illustrative stand-in reference state (NOT real silver labels)."""
    return uniform_random_latent_z(seed=seed)


all_pred, all_gold_binary, spearmans, top3_overlaps = [], [], [], []

for i, rec in enumerate(states):
    pred_vals = np.array([rec["pseudo_z"].mastery.values[k] for k in kc_ids])
    gold_z = toy_reference_z(seed=42 + i)
    gold_vals = np.array([gold_z.mastery.values[k] for k in kc_ids])
    gold_binary = (gold_vals > 0.5).astype(int)

    all_pred.extend(pred_vals.tolist())
    all_gold_binary.extend(gold_binary.tolist())

    rho, _ = spearmanr(pred_vals, gold_vals)
    spearmans.append(rho)

    top3_pred = set(np.argsort(-pred_vals)[:3].tolist())
    top3_gold = set(np.argsort(-gold_vals)[:3].tolist())
    overlap = len(top3_pred & top3_gold) / 3.0
    top3_overlaps.append(overlap)
    print(f"[{rec['dialogue_id']}] within-dialogue Spearman rho={rho:.3f}, top-3 overlap={overlap:.2f}")

print("\n--- pooled, n=3 toy dialogues (illustrative only) ---")
print(f"KC Brier           = {binary_brier(all_gold_binary, all_pred):.4f}")
print(f"KC ECE (5 bins)    = {ece(all_pred, all_gold_binary, n_bins=5):.4f}")
print(f"mean Spearman rho  = {np.nanmean(spearmans):.4f}")
print(f"mean top-3 overlap = {np.nanmean(top3_overlaps):.4f}")
print("\nCompare against results/expected_metrics.json for the real 394-dialogue numbers.")

## 4. The misconception sparsity caveat (the paper's key cautionary finding)

This is the most important methodological lesson in the manuscript, and it is
worth internalizing before looking at any misconception ranking metric.

On the real MathDial test set, **all 394 test dialogues share an identical
four-code active misconception set** (`M001`–`M004`, each at silver
probability 0.35) in silver label version v3. A trivial **label-prior
baseline** — always predict this fixed set, without ever reading the
dialogue — already achieves **F1@5 = 0.889**. The fine-tuned inverter reaches
F1@5 = 1.0 and MRR ≈ 1.0, but mostly by *learning this uniform template*, not
by discriminating genuine per-student misconception profiles.

**The methodological lesson:** near-perfect ranking metrics in a sparse-label
regime are meaningless without a label-prior baseline for comparison. Below,
we reproduce this exact comparison mechanically on the toy dialogues.

In [ ]:
from iss.eval.metrics import f1_at_k, mrr_at_k

# Toy "gold active set": pretend every toy dialogue activates the same 2
# misconception codes -- mirrors the real sparsity finding at small scale.
gold_active = {"M001", "M002"}
gold_binary = np.array([1 if m in gold_active else 0 for m in misc_ids])
label_prior_score = gold_binary.astype(float)  # always predicts the fixed active set

f1_prior, f1_model, mrr_prior, mrr_model = [], [], [], []
for rec in states:
    pred_score = np.array([rec["pseudo_z"].misconceptions.probs[m] for m in misc_ids])

    f1_model.append(f1_at_k(gold_binary, pred_score, k=5))
    f1_prior.append(f1_at_k(gold_binary, label_prior_score, k=5))

    positives = {i for i, g in enumerate(gold_binary) if g == 1}
    mrr_model.append(mrr_at_k(list(np.argsort(-pred_score)), positives, k=10))
    mrr_prior.append(mrr_at_k(list(np.argsort(-label_prior_score)), positives, k=10))

print("Toy misconception ranking, n=3 dialogues, gold active set = {M001, M002}")
print(f"  F1@5   dialogue-aware (Pseudo-Z) = {np.mean(f1_model):.3f}")
print(f"  F1@5   label-prior (fixed set)   = {np.mean(f1_prior):.3f}")
print(f"  MRR@10 dialogue-aware            = {np.mean(mrr_model):.3f}")
print(f"  MRR@10 label-prior               = {np.mean(mrr_prior):.3f}")
print("\nIf the label-prior baseline matches or beats the dialogue-aware score, high "
      "F1/MRR reflects label-set uniformity, not genuine per-dialogue discrimination -- "
      "exactly what scripts/robustness_misconception_sparsity.py audits at full scale.")

## 5. Where to go next

This notebook demonstrated the schema, state construction, and evaluation
*logic*. To reproduce the paper's actual numbers on the real MathDial corpus:

| What | Where |
|---|---|
| Download & preprocess MathDial | `scripts/download_data.py`, `scripts/build_dataset.py`, `scripts/build_splits.py` |
| Silver-Z labeling (LLM, needs API key) + deterministic V2→V3 post-processing | `scripts/label_latent_z_v2.py`, `scripts/postprocess_v2_to_v3.py` |
| QLoRA fine-tune the inverter / forward simulators (needs GPU) | `scripts/train_inverter.py`, `scripts/train_forward.py` |
| E1 inversion accuracy + BKT baseline | `scripts/run_inverter_eval.py`, `scripts/run_bkt_baseline.py` |
| E3 identifiability curve | `scripts/run_identifiability_inverter.py` |
| KC structure / misconception sparsity audits | `scripts/robustness_kc_structure.py`, `scripts/robustness_misconception_sparsity.py` |
| E4 counterfactual forward replay + statistics | `scripts/run_replay_eval.py`, `scripts/run_replay_statistics.py` |
| E5 label agreement, E7 Bridge probe | `scripts/compute_label_agreement_e5.py`, `scripts/run_e7_v3.py` |
| Verification targets | `results/expected_metrics.json` |

See the top-level `README.md` for the full command reference, or run
`python scripts/reproduce_results.py` to chain the stages that do not
require a not-yet-trained checkpoint.